# Measuring Bias: Fairness Metrics on a Toy Hiring Dataset

**CS474: Human Computer Interaction — Bias in Design**

"The algorithm decided" is never the end of the story: models trained on biased data reproduce that bias, and interfaces that surface their outputs inherit it.  The good news is that bias can often be *measured*.  In this notebook we build a tiny simulated resume-screening dataset and compute several standard fairness metrics.

You will:

1. Simulate a hiring dataset where a historical process disadvantaged one group
2. Train a *very* simple score-threshold "screener" on that data
3. Compute **selection rate**, **demographic parity difference**, the **four-fifths (80%) rule**, and **equal opportunity difference**
4. See why "the model never sees the group attribute" does *not* guarantee fairness

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(474)

## Part 1: Simulate Historical Hiring Data

Two groups, A and B, apply.  True qualification is *identically distributed* in both groups.  But the historical process embedded two problems:

- **Measurement bias:** group B's observed "score" (e.g., resume keywords favored by past reviewers) reads systematically 8 points lower than their true qualification.
- **Label bias:** past hiring decisions — our training labels — were made from those biased scores.

In [ ]:
N = 2000
group = rng.choice(['A', 'B'], size=N)
true_skill = rng.normal(70, 10, N)                    # same distribution for both groups

observed_score = true_skill + np.where(group == 'B', -8, 0) + rng.normal(0, 5, N)
hired_historically = observed_score > 75              # the biased historical labels

df = pd.DataFrame({'group': group, 'true_skill': true_skill,
                   'score': observed_score, 'hired': hired_historically})
df.groupby('group')[['true_skill', 'score', 'hired']].mean().round(2)

Notice: both groups have essentially the same **true skill**, but very different historical hiring rates.  Any model trained to imitate `hired` will learn this gap.

## Part 2: A "Neutral" Automated Screener

Suppose we now automate screening: select every applicant with `score > 75`.  The rule never mentions group membership — it is *facially neutral*.  Let's measure what it actually does.

In [ ]:
df['selected'] = df['score'] > 75

rates = df.groupby('group')['selected'].mean()
print("Selection rates:")
print(rates.round(3))

dp_diff = rates['A'] - rates['B']
ratio = rates['B'] / rates['A']
print(f"\nDemographic parity difference (A - B): {dp_diff:.3f}")
print(f"Disparate impact ratio (B / A):          {ratio:.3f}")
print(f"Four-fifths (80%) rule {'PASSES' if ratio >= 0.8 else 'FAILS'}"
      f" (US EEOC guideline: ratio should be at least 0.80)")

## Part 3: Equal Opportunity

Demographic parity compares raw selection rates.  **Equal opportunity** asks a sharper question: *among truly qualified applicants, are both groups selected at the same rate?*  (This is the difference in true positive rates.)  We'll define "truly qualified" as `true_skill > 75` — something we know only because this is a simulation.

In [ ]:
qualified = df[df['true_skill'] > 75]
tpr = qualified.groupby('group')['selected'].mean()
print("Selection rate among TRULY qualified applicants:")
print(tpr.round(3))
print(f"\nEqual opportunity difference (A - B): {tpr['A'] - tpr['B']:.3f}")

ax = tpr.plot.bar(color=['tab:blue', 'tab:orange'], rot=0, figsize=(4.5, 3))
ax.set_ylabel('P(selected | truly qualified)')
ax.set_title('Equal opportunity: qualified applicants selected')
ax.axhline(tpr.max(), ls=':', color='gray')
plt.show()

A qualified group-B applicant is far less likely to make it through — even though the screener "doesn't know" their group.  The bias rides in on the **proxy** (the score).

## Part 4: Visualize Where the Threshold Bites

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for g, color in [('A', 'tab:blue'), ('B', 'tab:orange')]:
    ax.hist(df.loc[df.group == g, 'score'], bins=40, alpha=0.55, label=f'group {g}', color=color)
ax.axvline(75, color='red', ls='--', label='selection threshold')
ax.set_xlabel('observed score'); ax.set_ylabel('applicants')
ax.set_title('Same true skill, shifted observed scores')
ax.legend()
plt.show()

## Your Turn

1. **Tune the bias.**  Reduce the measurement bias from `-8` to `-3`, then to `0`.  At what point does the four-fifths rule pass?  Does equal opportunity reach zero at the same point?
2. **"Fix" it with a threshold?**  Try using per-group thresholds so that selection rates match.  What new controversy does this create?  (This is a real policy debate — there is no purely technical answer.)
3. **Design connection.**  You are designing the *interface* that recruiters use with this screener.  Propose two UI changes (e.g., what is displayed next to each recommendation, what actions are afforded) that would help a human catch this bias rather than rubber-stamp it.

## Reflection

The metrics here — demographic parity, disparate impact, equal opportunity — are the same ones used to audit real systems (see the ProPublica COMPAS investigation and *Dissecting racial bias in an algorithm used to manage the health of populations*, Science 2019, from our course readings).  Measurement doesn't resolve the ethical question, but it turns "seems unfair" into evidence you can act on.